# Words-per-Page Calibration — *The Moon Race* book

**Exercise 03 · CrewAI LaTeX Book Generator** — research / results analysis (Guidelines V3 §9).

**Question.** How many words of prose does this figure-rich book need to land in the
PRD's **13–17 page** window (target 15)? The naive 450 words/page assumption is wrong
here, because the cover, table of contents, one figure per chapter, a table, an
equation, a Python plot, the TikZ appendix, the bibliography, and a page break after
every chapter add a large *fixed* page overhead.

**Method (one-factor-at-a-time).** We held the layout constant and varied a single
parameter — the amount of prose — by repeating each section's paragraphs 1×, 2×, and
3×, compiled each variant with `latexmk`/`pdflatex`, and counted pages with `pypdf`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Measured: same layout, paragraphs repeated 1x / 2x / 3x, pages via pypdf.
words = np.array([1631, 3262, 4893])
pages = np.array([16, 17, 21])
for w, p in zip(words, pages, strict=True):
    print(f'{w:>5} words -> {p:>2} pages  (effective {w / p:6.1f} words/page)')

In [ ]:
# Affine fit: pages = overhead + words / density
slope, intercept = np.polyfit(words, pages, 1)
density = 1 / slope
print(f'pages ≈ {intercept:.1f} + words / {density:.0f}')
print(f'Fixed overhead : {intercept:.1f} pages (front matter, figures, table, eq, plot, appendix, bib, breaks)')
print(f'Text density   : {density:.0f} words per added page')

target = 15
need = (target - intercept) * density
print(f'Words to hit {target} pages: ~{need:.0f}')
print(f'Current book   : {words[0]} words -> {pages[0]} pages (inside PRD 13-17)')

In [ ]:
xs = np.linspace(1000, 5200, 100)
ys = intercept + xs / density
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.axhspan(13, 17, color='#27ae60', alpha=0.12, label='PRD target 13-17')
ax.axhline(15, color='#27ae60', ls='--', lw=1)
ax.plot(xs, ys, color='#2980b9', label=f'fit: pages ≈ {intercept:.1f} + words/{density:.0f}')
ax.scatter(words, pages, color='#c0392b', zorder=3, label='measured runs')
for w, p in zip(words, pages, strict=True):
    ax.annotate(f'{w}w, {p}p', (w, p), textcoords='offset points', xytext=(8, -4), fontsize=9)
ax.set_xlabel('Prose words in book')
ax.set_ylabel('Compiled PDF pages')
ax.set_title('Words-per-page calibration - Moon Race book')
ax.legend()
fig.tight_layout()
fig.savefig('words_per_page_calibration.png', dpi=130)
print('saved words_per_page_calibration.png')

## Findings

* The relationship is **affine, not proportional**: about **13 fixed pages** of overhead
  plus **one page per ~650 words** of prose. The book is overhead-dominated.
* Therefore *effective* words/page rises with length (102 → 192 → 233). A single divisor
  is only valid near the operating point — a useful sensitivity result.
* **Update (subject-page hard rule):** the requirement was later tightened to
  **≥15 *subject* pages** (chapters alone, excluding cover/TOC/appendix/references).
  Measured at the new operating point: 4,004 words → 16 subject pages (canonical) and
  5,038 words → 16 subject pages (live Opus run), i.e. **~250–300 prose words per
  subject page**. `config/book.json` therefore now sets `words_per_page = 280`, and the
  review gate estimates *subject* pages.
* **Recommendation:** keep prose in the **~4,200–5,100 word** band to clear the 15
  subject-page floor; the compiled book lands at ≈19–21 sheets total.